# 05 - Inference & Deployment

Loads the final joint checkpoint (`mtl_best.pt`) and:
1. Runs single-image inference (classification + boxes + mask) end-to-end.
2. Exports a TorchScript version for fast, dependency-light deployment.
3. Launches a Gradio demo (shareable link) so you can try it from a browser / share it with others.

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from src import config
from src.inference import load_model, predict
from src.utils.visualize import show_image_mask, show_image_boxes

CHECKPOINT = "/kaggle/working/multi-task-chest-xray-analysis-system/mtl_best.pt"
model = load_model(CHECKPOINT, device=config.DEVICE)
print("Model loaded on", config.DEVICE)


## 1. Run inference on a sample image

Works with `.jpg`, `.png`, or raw `.dcm` files.

In [ ]:
# BUG FIX: was missing the "competitions/" path segment that Kaggle
# actually mounts this dataset under (see notebook 02's DET_IMG_PATH),
# which would raise a pydicom FileNotFoundError.
SAMPLE_IMAGE = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images/0004cfab-14fd-4e49-80ba-63a80b6bddd6.dcm"

result = predict(model, SAMPLE_IMAGE, conf_threshold=0.3, seg_threshold=0.5)
print("Predicted class:", result["predicted_class"])
print("Class probabilities:", result["class_probs"])
print("Detected boxes:", result["boxes"])

show_image_mask(result["image"], pred_mask=result["mask_binary"], title=result["predicted_class"])
show_image_boxes(result["image"], [b["box"] for b in result["boxes"]],
                 [b["score"] for b in result["boxes"]], title=result["predicted_class"])


## 2. Export to TorchScript (portable, no Python class definitions needed to reload)

In [ ]:
model.eval()
example = torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE).to(config.DEVICE)

traced = torch.jit.trace(model, example)
# BUG FIX: was missing the leading "/" - this was silently saving to a
# relative path (likely to fail with FileNotFoundError, or land somewhere
# unexpected) despite the print statement claiming the absolute path below.
traced.save("/kaggle/working/multi-task-chest-xray-analysis-system/multichexnet_traced.pt")
print("Saved TorchScript model to /kaggle/working/multi-task-chest-xray-analysis-system/multichexnet_traced.pt")

# Reload check (no need to import MultiCheXNet class to use this file elsewhere):
reloaded = torch.jit.load("/kaggle/working/multi-task-chest-xray-analysis-system/multichexnet_traced.pt", map_location=config.DEVICE)
with torch.no_grad():
    out = reloaded(example)
print({k: v.shape for k, v in out.items()})


## 3. Launch a Gradio demo

This opens an interactive web UI. In Colab, pass `share=True` to get a public link you can send to anyone (valid ~72h).

In [ ]:
sys.path.insert(0, f"{PROJECT_DIR}/app")
from gradio_app import build_app

demo = build_app(CHECKPOINT, device=config.DEVICE)
demo.launch(share=True, debug=False)


## 4. (Optional) Save everything to Google Drive for persistence

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy("/content/mtl_best.pt", "/content/drive/MyDrive/mtl_best.pt")
# shutil.copy("/content/multichexnet_traced.pt", "/content/drive/MyDrive/multichexnet_traced.pt")


### That's it!

You now have:
- `mtl_best.pt` - full PyTorch state_dict checkpoint (needs the `MultiCheXNet` class to reload)
- `multichexnet_traced.pt` - self-contained TorchScript model (no class definition needed)
- A working Gradio demo you can embed anywhere or share via the public link

See the project `README.md` for how to serve `multichexnet_traced.pt` behind a simple FastAPI endpoint if you want a "real" backend instead of Gradio.